In [11]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

In [12]:
import torch
from src.config import DATASET_ROOT, POSE_DATASET_ROOT
from src.Skeleton_model.yolo_pose_tracking import save_annotated_pose_videos
from src.rwf2000 import RWF2000PoseDataset, RWF2000Dataset
from src.Skeleton_model.graph import SkeletonGraph, compute_joint_distance_to_center_of_gravity
from src.Skeleton_model.stgcn import STGCN
from scripts.common.get_device import get_available_device
from ultralytics import YOLO

pose_dataset = RWF2000PoseDataset(POSE_DATASET_ROOT, split="train")
radii = compute_joint_distance_to_center_of_gravity(pose_dataset)
skeleton_graph = SkeletonGraph(radii)
device = get_available_device()
model = STGCN(adjacency=skeleton_graph.A).to(device)

Using cuda:4 with 23.35 GB free


/mnt/vurm/homes/homes/mp2940/violence-detection-dissertation/src/Skeleton_model/graph.py:54: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.radii = torch.tensor(radii, dtype=torch.float32)


In [13]:
model.train()
model.zero_grad(set_to_none=True)

x, label = pose_dataset[0]
x = x.unsqueeze(0).to(device)
target = torch.tensor([label], device=device)

output = model(x)
loss = F.cross_entropy(output, target)

loss.backward()

print("Loss:", loss.item())

Loss: 0.5855059623718262


In [14]:
x, label = pose_dataset[0]
x = x.unsqueeze(0).to(device)
output = model(x)

print("Input shape: ", x.shape)
print("Output shape:", output.shape)
print(output)

assert output.shape == (1, 2)
assert torch.isfinite(output).all()

print("Forward-pass test passed.")

Input shape:  torch.Size([1, 3, 150, 17, 2])
Output shape: torch.Size([1, 2])
tensor([[ 0.1251, -0.1205]], device='cuda:4', grad_fn=<ViewBackward0>)
Forward-pass test passed.


In [8]:
import torch.nn.functional as F


model.train()

target = torch.tensor([label], device=device)

loss = F.cross_entropy(output, target)
loss.backward()

print("Loss:", loss.item())
print("Backward pass successful!")

Loss: 0.5124380588531494
Backward pass successful!


In [9]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"{name:40} {param.grad is not None}")

data_batch_norm.weight                   True
data_batch_norm.bias                     True
stgcn_blocks.0.spatial_graph_conv.channel_transform.weight True
stgcn_blocks.0.spatial_graph_conv.channel_transform.bias True
stgcn_blocks.0.temporal_conv.0.weight    True
stgcn_blocks.0.temporal_conv.0.bias      True
stgcn_blocks.0.temporal_conv.2.weight    True
stgcn_blocks.0.temporal_conv.3.weight    True
stgcn_blocks.0.temporal_conv.3.bias      True
stgcn_blocks.1.spatial_graph_conv.channel_transform.weight True
stgcn_blocks.1.spatial_graph_conv.channel_transform.bias True
stgcn_blocks.1.temporal_conv.0.weight    True
stgcn_blocks.1.temporal_conv.0.bias      True
stgcn_blocks.1.temporal_conv.2.weight    True
stgcn_blocks.1.temporal_conv.3.weight    True
stgcn_blocks.1.temporal_conv.3.bias      True
stgcn_blocks.2.spatial_graph_conv.channel_transform.weight True
stgcn_blocks.2.spatial_graph_conv.channel_transform.bias True
stgcn_blocks.2.temporal_conv.0.weight    True
stgcn_blocks.2.temporal_

In [10]:
num_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Parameters: {num_params:,}")
print(f"Trainable: {trainable:,}")

Parameters: 3,072,006
Trainable: 3,072,006


In [ ]:
from Skeleton_model.stgcn import SpatialGraphConv
x, label = pose_dataset[0]
adjacency = skeleton_graph.A 

batch_size = 1
in_channels = 3
out_channels = 64
num_partitions = 3

spatial_conv = SpatialGraphConv(
    in_channels=in_channels,
    out_channels=out_channels,
    num_partitions=num_partitions,
)

C, T, V, M = x.shape
x = x.permute(3, 0, 1, 2)
x = x.reshape(batch_size*M, C, T, V)

# Forward pass
output = spatial_conv(x, adjacency)

print("Input shape :", x.shape)
print("Output shape:", output.shape)


Input shape : torch.Size([2, 3, 150, 17])
Output shape: torch.Size([2, 64, 150, 17])


In [12]:
print(torch.isnan(output).any())
print(torch.isinf(output).any())

tensor(False)
tensor(False)


In [4]:
print(output.mean())
print(output.std())
print(output.min())
print(output.max())

tensor(0.0450, grad_fn=<MeanBackward0>)
tensor(0.3222, grad_fn=<StdBackward0>)
tensor(-1.0387, grad_fn=<MinBackward1>)
tensor(1.2728, grad_fn=<MaxBackward1>)


In [9]:
with torch.no_grad():
    transformed = spatial_conv.channel_transform(x)

print("After channel transform:")
print("Mean:", transformed.mean())
print("Std: ", transformed.std())

print("\nAfter graph propagation:")
print("Mean:", output.mean())
print("Std: ", output.std())

After channel transform:
Mean: tensor(0.0441)
Std:  tensor(0.4589)

After graph propagation:
Mean: tensor(0.0450, grad_fn=<MeanBackward0>)
Std:  tensor(0.3222, grad_fn=<StdBackward0>)


In [10]:
print("Adjacency min:", adjacency.min())
print("Adjacency max:", adjacency.max())
print("Partition sums:", adjacency.sum(dim=(1, 2)))

Adjacency min: tensor(0.)
Adjacency max: tensor(0.5000)
Partition sums: tensor([5.7333, 5.2667, 6.0000])


In [11]:
print(x.min(), x.max(), x.mean(), x.std())

tensor(0.) tensor(1.) tensor(0.3474) tensor(0.3717)


In [17]:
joint_names = [
    "nose", "left_eye", "right_eye", "left_ear", "right_ear",
    "left_shoulder", "right_shoulder",
    "left_elbow", "right_elbow",
    "left_wrist", "right_wrist",
    "left_hip", "right_hip",
    "left_knee", "right_knee",
    "left_ankle", "right_ankle",
]

for name, distance in zip(joint_names, radii):
    print(f"{name:15s}: {distance:.4f}")

nose           : 0.1240
left_eye       : 0.1311
right_eye      : 0.1333
left_ear       : 0.1406
right_ear      : 0.1422
left_shoulder  : 0.0995
right_shoulder : 0.0997
left_elbow     : 0.0609
right_elbow    : 0.0612
left_wrist     : 0.0651
right_wrist    : 0.0667
left_hip       : 0.0595
right_hip      : 0.0594
left_knee      : 0.1307
right_knee     : 0.1312
left_ankle     : 0.2052
right_ankle    : 0.2053
